第一輪品質檢查

In [1]:
# 第一步:讀取資料
import pandas as pd

df = pd.read_excel(
    "../data/raw/Online Retail.xlsx",
    engine="openpyxl"
)

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [2]:
# 第二步:確認資料規模(確認是否有少資料)
print(f"資料筆數: {df.shape[0]}，欄位數: {df.shape[1]}")

資料筆數: 541909，欄位數: 8


In [3]:
# 第三步:確認欄位型態(各欄位是否有資料缺失)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


In [4]:
# 第四步:空值分析(確認缺失值占比有多少, 對未來分析是否有影響)
df.isnull().sum()
round(df.isnull().sum() / len(df) * 100, 2)

InvoiceNo       0.00
StockCode       0.00
Description     0.27
Quantity        0.00
InvoiceDate     0.00
UnitPrice       0.00
CustomerID     24.93
Country         0.00
dtype: float64

In [5]:
# 第五步:重複資料(清理時要確認要不要刪除)
df.duplicated().sum()

np.int64(5268)

In [6]:
# 第六步:日期資料
print(f"資料最早日期: {df['InvoiceDate'].min()}，資料最晚日期: {df['InvoiceDate'].max()}")

資料最早日期: 2010-12-01 08:26:00，資料最晚日期: 2011-12-09 12:50:00


In [7]:
# 第七步:數量異常(初步可能判定資料異常, 但不一定是錯誤資料, 可能是退貨)
(df['Quantity'] <= 0).sum()
df['Quantity'].describe()

count    541909.000000
mean          9.552250
std         218.081158
min      -80995.000000
25%           1.000000
50%           3.000000
75%          10.000000
max       80995.000000
Name: Quantity, dtype: float64

In [8]:
# 第八步:價格異常(初步可能判定資料異常, 但不一定是錯誤資料, 可能是退貨)
(df['UnitPrice'] <= 0).sum()
df['UnitPrice'].describe()

count    541909.000000
mean          4.611114
std          96.759853
min      -11062.060000
25%           1.250000
50%           2.080000
75%           4.130000
max       38970.000000
Name: UnitPrice, dtype: float64

In [9]:
# 第九步:取消訂單
(df['InvoiceNo'].str.startswith('C')).sum()

9288

第二輪品質檢查, 針對數量/價格/重複資料 確認合理性, 異常偵測

In [10]:
(df['Quantity'] < 0).sum()

np.int64(10624)

In [11]:
(df['Quantity'] == 0).sum()

np.int64(0)

In [12]:
(df['UnitPrice'] < 0).sum()

np.int64(2)

In [13]:
(df['UnitPrice'] == 0).sum()

np.int64(2515)

In [38]:
duplicate_df = df[df.duplicated(keep=False)]
duplicate_df.head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
548,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920.0,United Kingdom
555,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920.0,United Kingdom


In [36]:
duplicate_df.groupby(list(df.columns)).size().value_counts()
duplicate_df.shape

(10147, 8)

In [14]:
# 未來可做地區分析
df['Country'].nunique()

38

In [15]:
# 未來可做會員分析
df['CustomerID'].nunique()

4372

第三輪檢查, 異常原因分析

In [16]:
df[df['UnitPrice'] < 0]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


In [17]:
df[df['UnitPrice'] == 0].head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
1988,536550,85044,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2024,536552,20950,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2025,536553,37461,NaN,3,2010-12-01 14:35:00,0.0,NaN,United Kingdom
2026,536554,84670,NaN,23,2010-12-01 14:35:00,0.0,NaN,United Kingdom
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom


In [18]:
df[df['Quantity'] < 0].head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
238,C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
239,C536391,21484,CHICK GREY HOT WATER BOTTLE,-12,2010-12-01 10:24:00,3.45,17548.0,United Kingdom
240,C536391,22557,PLASTERS IN TIN VINTAGE PAISLEY,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
241,C536391,22553,PLASTERS IN TIN SKULLS,-24,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
939,C536506,22960,JAM MAKING SET WITH JARS,-6,2010-12-01 12:38:00,4.25,17897.0,United Kingdom


In [19]:
# 零價格紀錄分類摘要
zero_price = df[df["UnitPrice"] == 0].copy()

In [20]:
zero_price["Description"].isna().sum()

np.int64(1454)

In [21]:
zero_price["CustomerID"].isna().sum()

np.int64(2475)

In [22]:
(zero_price["Quantity"] > 0).sum()

np.int64(1179)

In [23]:
(zero_price["Quantity"] < 0).sum()

np.int64(1336)

In [24]:
zero_price["Description"].value_counts().head(20)

Description
check                              159
?                                   47
damages                             45
damaged                             43
found                               25
sold as set on dotcom               20
adjustment                          16
Damaged                             14
FRENCH BLUE METAL DOOR SIGN 1        9
thrown away                          9
Unsaleable, destroyed.               9
amazon                               8
FRENCH BLUE METAL DOOR SIGN 8        8
Found                                8
FRENCH BLUE METAL DOOR SIGN 4        7
FRENCH BLUE METAL DOOR SIGN No       7
OWL DOORSTOP                         7
FRENCH BLUE METAL DOOR SIGN 3        7
RECIPE BOX PANTRY YELLOW DESIGN      7
??                                   7
Name: count, dtype: int64

In [25]:
# 交叉驗證 10,624 筆 Quantity < 0，是否真的都可以用 InvoiceNo 的 C 開頭
negative_qty = df[df["Quantity"] < 0]
negative_qty["InvoiceNo"].astype(str).str.startswith("C").value_counts()

InvoiceNo
True     9288
False    1336
Name: count, dtype: int64

In [26]:
# 驗證 false 的 1336 筆 是否等於 UnitPrice = 0
# special_case = negative_qty[(df["Quantity"] < 0) & ~negative_qty["InvoiceNo"].astype(str).str.startswith("C")]
special_case = negative_qty[~negative_qty["InvoiceNo"].astype(str).str.startswith("C")]
special_case.head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom
4347,536764,84952C,NaN,-38,2010-12-02 14:42:00,0.0,NaN,United Kingdom
7188,536996,22712,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom
7189,536997,22028,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom
7190,536998,85067,NaN,-6,2010-12-03 15:30:00,0.0,NaN,United Kingdom
7192,537000,21414,NaN,-22,2010-12-03 15:32:00,0.0,NaN,United Kingdom
7193,537001,21653,NaN,-6,2010-12-03 15:33:00,0.0,NaN,United Kingdom
7195,537003,85126,NaN,-2,2010-12-03 15:33:00,0.0,NaN,United Kingdom
7196,537004,21814,NaN,-30,2010-12-03 15:34:00,0.0,NaN,United Kingdom
7197,537005,21692,NaN,-70,2010-12-03 15:35:00,0.0,NaN,United Kingdom


In [27]:
special_case["UnitPrice"].value_counts().head(20)

UnitPrice
0.0    1336
Name: count, dtype: int64

In [28]:
special_case["Description"].value_counts().head(20)

Description
check                         120
damages                        45
damaged                        42
?                              41
sold as set on dotcom          20
Damaged                        14
thrown away                     9
Unsaleable, destroyed.          9
??                              7
damages?                        5
wet damaged                     5
ebay                            5
smashed                         4
missing                         3
CHECK                           3
wet pallet                      3
Dotcom sales                    2
reverse 21/5/10 adjustment      2
counted                         2
crushed                         2
Name: count, dtype: int64

In [47]:
# 調查資料重複群組結構
# 1. 計算每個完整重複群組出現幾次
duplicate_group_sizes = (df.groupby(list(df.columns), dropna=False).size().reset_index(name="duplicate_count"))

duplicate_group_sizes[duplicate_group_sizes["duplicate_count"] > 1]["duplicate_count"].value_counts().sort_index()

duplicate_count
2     4588
3      238
4       39
5        5
6        6
8        1
12       1
20       1
Name: count, dtype: int64

In [48]:
# 2. 查看重複最多的群組
duplicate_group_sizes[duplicate_group_sizes["duplicate_count"] > 1].sort_values(by="duplicate_count", ascending=False).head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,duplicate_count
207580,555524,22698,PINK REGENCY TEACUP AND SAUCER,1,2011-06-05 11:37:00,2.95,16923.0,United Kingdom,20
207579,555524,22697,GREEN REGENCY TEACUP AND SAUCER,1,2011-06-05 11:37:00,2.95,16923.0,United Kingdom,12
408646,572861,22775,PURPLE DRAWERKNOB ACRYLIC EDWARDIAN,12,2011-10-26 12:46:00,1.25,14102.0,United Kingdom,8
403109,572344,M,Manual,48,2011-10-24 10:43:00,1.50,14607.0,United Kingdom,6
481780,578289,23395,BELLE JARDINIERE CUSHION COVER,1,2011-11-23 14:07:00,3.75,17841.0,United Kingdom,6
48190,540524,21756,BATH BUILDING BLOCK WORD,1,2011-01-09 12:53:00,5.95,16735.0,United Kingdom,6
57332,541266,21754,HOME BUILDING BLOCK WORD,1,2011-01-16 16:25:00,5.95,15673.0,United Kingdom,6
25692,538514,21756,BATH BUILDING BLOCK WORD,1,2010-12-12 14:27:00,5.95,15044.0,United Kingdom,6
57333,541266,21755,LOVE BUILDING BLOCK WORD,1,2011-01-16 16:25:00,5.95,15673.0,United Kingdom,6
25770,538517,22749,FELTCRAFT PRINCESS CHARLOTTE DOLL,1,2010-12-12 16:05:00,3.75,17618.0,United Kingdom,5


In [49]:
# 3. 確認重複資料是否包含取消或非銷售紀錄
duplicate_df_non_sales = duplicate_df.copy()

duplicate_df_non_sales["is_cancelled"] = duplicate_df_non_sales["InvoiceNo"].astype(str).str.startswith("C")

print ("重複資料中，取消訂單筆數: ", duplicate_df_non_sales['is_cancelled'].sum())
print ("重複資料中，非銷售紀錄筆數: ", (duplicate_df_non_sales['UnitPrice'] <= 0).sum())

重複資料中，取消訂單筆數:  69
重複資料中，非銷售紀錄筆數:  10
